In [ ]:
from pathlib import Path
from typing import List, Tuple
from random import randint
from PIL import Image
import pandas as pd
from transformers import CLIPModel, CLIPProcessor

In [ ]:
csv = (
    pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv"))
    .drop("no", axis=1)
    .assign(image=lambda x: x.image.astype(str))
)
csv

In [ ]:
columns = csv.columns[2:]
columns

In [ ]:
def load_processor_model(
    processor_name: str, model_name: str
) -> Tuple[CLIPProcessor, CLIPModel]:
    processor = CLIPProcessor.from_pretrained(processor_name)
    model = CLIPModel.from_pretrained(model_name)
    return processor, model


processor, model = load_processor_model(
    "openai/clip-vit-base-patch32", "openai/clip-vit-base-patch32"
)

In [ ]:
def get_similarity_scores(class_items: List[str], image: Image) -> List[float]:
    inputs = processor(
        text=class_items,
        images=[image],
        return_tensors="pt",  # pytorch tensors
    )
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    class_likelihoods = logits_per_image.softmax(dim=1).detach().numpy()
    return class_likelihoods[0]

In [ ]:
row = csv.iloc[randint(0, len(csv))]
pil_img = Image.open(
    Path(".")
    .joinpath("data")
    .joinpath("images")
    .joinpath(row["image"])
    .with_suffix(".jpg")
)
pil_img

In [ ]:
class_likelihoods = get_similarity_scores(columns, pil_img)

res_data = {"label": [], "prediction": [], "ground truth": []}

for class_item, class_likelihood in zip(columns, class_likelihoods):
    res_data["label"].append(class_item)
    res_data["prediction"].append(f"{class_likelihood:.2%}")
    res_data["ground truth"].append(row[class_item])

pd.DataFrame(data=res_data)